In [5]:
import pandas as pd
from tools import hash_danych_bezpieczny

In [6]:
# TODO: opcjonalnie po wersji MVP, jako jedna z poprawek - aktualizacja tabeli towarów (więcej SKU)
# df_tow = pd.read_csv(
#     "dane/Towar_2026.csv",
#     encoding='utf-8-sig',  # obsługuje BOM
#     sep=';',
#     decimal=",",
#     on_bad_lines='skip'
# )
# print(df_towar_new.shape)
# print(df_towar_new['TowId'].max())
# print(df_towar_new[df_towar_new['TowId'] > 81360]['TowId'].nunique())
# df_towar_new[df_towar_new['TowId']>81360]['Nazwa']

In [ ]:
INTERIM_PATH = "dane/interim/"

In [8]:
hashe_wejsciowe = [
    ('towar_selected', ['TowId']),
    ('asort_selected', ['AsId']),
    ('dok_selected', ['DokId']),
    ('pozdok_selected', ['DokId', 'TowId'])
]

for nazwa, kolumny in hashe_wejsciowe:
    h = hash_danych_bezpieczny(f"{INTERIM_PATH}{nazwa}.parquet", kolumny_sortowania=kolumny)
    print(f"Hash {nazwa}: {h}")

Hash towar_selected: d1fad1663d3c44c76bb2fe55f714086a3a9467916c80105c7be41d05b531465b
Hash asort_selected: ff7753ffbfa1693da75e1266aac6b741340eb8f84d95c76ef92cd366306784a4
Hash dok_selected: 064df544d3f090de325d40d9782c23317c98831fd67a89728eba19343960649d
Hash pozdok_selected: a8aa53e2e6f735bccc00f67465418ad0e1173402f8bc85fc9c0f75291572cdca


**Hash towar_selected:** d1fad1663d3c44c76bb2fe55f714086a3a9467916c80105c7be41d05b531465b  
**Hash asort_selected:** ff7753ffbfa1693da75e1266aac6b741340eb8f84d95c76ef92cd366306784a4  
**Hash dok_selected:** 064df544d3f090de325d40d9782c23317c98831fd67a89728eba19343960649d  
**Hash pozdok_selected:** a8aa53e2e6f735bccc00f67465418ad0e1173402f8bc85fc9c0f75291572cdca  

In [10]:

df_tow = pd.read_parquet(f"{INTERIM_PATH}towar_selected.parquet")
df_asort = pd.read_parquet(f"{INTERIM_PATH}asort_selected.parquet")
df_dok = pd.read_parquet(f"{INTERIM_PATH}dok_selected.parquet")
df_pozd = pd.read_parquet(f"{INTERIM_PATH}pozdok_selected.parquet")

print(df_tow.shape, df_asort.shape, df_dok.shape, df_pozd.shape)

(41378, 7) (337, 2) (952122, 9) (4272656, 9)


In [11]:
print("df_tow TowId:", df_tow['TowId'].dtype)
print("df_dok DokId:", df_dok['DokId'].dtype)
print("df_pozd TowId:", df_pozd['TowId'].dtype)
print("df_pozd DokId:", df_pozd['DokId'].dtype)
print("df_asort AsId:", df_asort['AsId'].dtype)
print("df_tow AsId:", df_tow['AsId'].dtype)

df_tow TowId: int64
df_dok DokId: int64
df_pozd TowId: int64
df_pozd DokId: int64
df_asort AsId: int64
df_tow AsId: int64


In [12]:
# Sprawdzenie duplikatow na kluczach laczenia (ryzyko fan-out przy merge)
print("Duplikaty DokId w df_dok:", df_dok['DokId'].duplicated().sum())
print("Duplikaty TowId w df_tow:", df_tow['TowId'].duplicated().sum())
print("Duplikaty AsId w df_asort:", df_asort['AsId'].duplicated().sum())

Duplikaty DokId w df_dok: 0
Duplikaty TowId w df_tow: 0
Duplikaty AsId w df_asort: 0


In [13]:
# Krok 1: PozDok + Dok
dok_pozd = df_pozd.merge(df_dok, on='DokId', how='inner')
print(f"dok_pozd: {dok_pozd.shape}")

# Krok 2: + Towar
dok_pozd_tow = dok_pozd.merge(df_tow, on='TowId', how='inner')
print(f"dok_pozd_tow: {dok_pozd_tow.shape}")

# Krok 3: + Asort
fact_inka = dok_pozd_tow.merge(df_asort, on='AsId', how='inner')
print(f"fact_inka: {fact_inka.shape}")

dok_pozd: (4272656, 17)
dok_pozd_tow: (4261678, 23)
fact_inka: (4261678, 24)


In [14]:
# Diagnostyka utraconych wierszy przy inner join (porownanie z left join + indicator)
check1 = df_pozd.merge(df_dok, on='DokId', how='left', indicator=True)
print("Krok 1 (PozDok+Dok) - bez dopasowania:", (check1['_merge'] == 'left_only').sum())

check2 = dok_pozd.merge(df_tow, on='TowId', how='left', indicator=True)
print("Krok 2 (+Towar) - bez dopasowania:", (check2['_merge'] == 'left_only').sum())

check3 = dok_pozd_tow.merge(df_asort, on='AsId', how='left', indicator=True)
print("Krok 3 (+Asort) - bez dopasowania:", (check3['_merge'] == 'left_only').sum())


Krok 1 (PozDok+Dok) - bez dopasowania: 0
Krok 2 (+Towar) - bez dopasowania: 10978
Krok 3 (+Asort) - bez dopasowania: 0


In [15]:
# Analiza brakujacych TowId w Kroku 2 (dok_pozd + Towar)
missing_tow_ids = check2[check2['_merge'] == 'left_only']['TowId'].unique()
print("Liczba unikalnych brakujacych TowId:", len(missing_tow_ids))
print(missing_tow_ids[:20])

# ile transakcji/wartosci to dotyczy
print("Suma Wartosc dla utraconych wierszy:", check2[check2['_merge'] == 'left_only']['Wartosc'].sum())


Liczba unikalnych brakujacych TowId: 573
[81955 81745 81945 81365 81628 81361 81373 81362 81413 81426 81428 81508
 81548 81384 81461 81415 81852 81856 81363 81364]
Suma Wartosc dla utraconych wierszy: 116627.2019


In [16]:
doc_type_map = pd.DataFrame([
    (2,   'PZ',             1, True,  'IP',           +1, 'przyjecie', False),
    (8,   'ZWPAR',          1, True,  'IP',           +1, 'zwrot',     False),
    (9,   'PW',             1, True,  'IP',           +1, 'przyjecie', False),  # ← NOWY jak PZ
   
    (10,  'RW',             4, True,  'IP',           -1, 'rozchod',   False),
    (21,  'DF',             4, True,  'IP',           -1, 'sprzedaz',  False),
    (23,  'ST',             4, True,  'IP',           -1, 'strata',    True),
    
    (14,  'BO',             3, True,  'RESET_IP',     +1, 'bo',        False),
    (16,  'REM',            3, True,  'RESET_IP',     +1, 'remanent',  False),

    (26,  'ROZB',           1, True,  'IP_IM_DELTA',  +1, 'rozbieznosc',      False),
    (78,  'PRZES',          1, True,  'IP_IM_DELTA',  +1, 'przesuniecie',     False),

    (88,  'PRZES_GR',       2, True,  'IP_IM_DELTA',  +1, 'przesuniecie',     False),
    
    (1,   'OPAK',           2, False, 'Brak',         0, 'neutralny', False),
    (4,   'ZWFD',           4, False, 'Brak',         0, 'neutralny', False),
    (18,  'PRZEC',          6, False, 'Brak',         0, 'przecena',  True),
    (19,  'ZAMR_PRZEC',     0, False, 'Brak',         0, 'przecena',  True),
    (30,  'B_OPAK',         0, False, 'Brak',         0, 'neutralny', False),
    (33,  'FV',             0, False, 'Brak',         0, 'neutralny', False),
    (50,  'ZAM',            0, False, 'Brak',         0, 'neutralny', False),
    (59,  'DF',             0, False, 'Brak',         0, 'neutralny', False),
    (60,  'XXX',            0, False, 'Brak',         0, 'neutralny', False),
    (81,  'CDPRS',          1, False, 'Brak',         0, 'prasa',     False),
    (82,  'CRPRS',          7, False, 'Brak',         0, 'prasa',     False),
    (100, 'ZM_ST_VAT',      1, False, 'Brak',         0, 'neutralny', False),
    (100, 'ZM_ST_VAT',      4, False, 'Brak',         0, 'neutralny', False),
    (126, 'PLAN_ZM_ST_VAT', 0, False, 'Brak',         0, 'neutralny', False),
    (900, 'ST_rozli',       4, False, 'Brak',         0, 'neutralny', False),
    (900, 'ZWPAR_rozli',    1, False, 'Brak',         0, 'neutralny', False),
    (981, 'MM_INW_MOB',     0, False, 'Brak',         0, 'neutralny', False),  # ← NOWY
], columns=['TypDok', 'Dokument', 'TypPoz', 'WplywNaStan',
            'MetodaLiczenia', 'Mnoznik', 'TypRuchu', 'CzyNiechciane'])

In [17]:
fact_inka_merge = fact_inka.merge(doc_type_map, on=['TypDok', 'TypPoz'], how='left')

In [18]:
fact_inka_merge_data = fact_inka_merge[fact_inka_merge['Data'].dt.year.isin([2023, 2024, 2025, 2026])]
print(f"Po filtrze dat: {fact_inka_merge_data.shape}")

Po filtrze dat: (4260543, 30)


In [19]:
fact_inka_merge_data.to_parquet(f"{INTERIM_PATH}fact_inka.parquet", compression='zstd',index=False)
print(f"Zapisano: {fact_inka_merge_data.shape}")

Zapisano: (4260543, 30)


In [21]:
hashe_wyjsciowe = [
    ('fact_inka', ['TowId', 'Data'])
]

for nazwa, kolumny in hashe_wyjsciowe:
    h = hash_danych_bezpieczny(f"{INTERIM_PATH}{nazwa}.parquet", kolumny_sortowania=kolumny)
    print(f"Hash {nazwa}: {h}")

Hash fact_inka: 012e676339f1c152c164628fdf5fa40ba7751394ae3728f2cbd7dfb54e15de56


#### Hash fact_inka: 012e676339f1c152c164628fdf5fa40ba7751394ae3728f2cbd7dfb54e15de56